# THREE_MATERIAL process bridge — final efficient implementation

This notebook preserves the frozen P0 equilibrium semantics and adds one process-bridge implementation for IRA-900-C, MIL-101(Cr)-TEPA(30), and Al2O3-TEPA(20).

Quantities are kept distinct: `H`, `M=E[q]`, regeneration-referenced `W_eq`, finite-time `W_cyc`, and optimized mass-specific productivity `P_star`.

Hard stops: no arbitrary q-reg threshold; no kinetic extrapolation outside measured temperature support; no RH-kinetic surface invented from sparse humid/dry data; no fixed-policy productivity ranking duplication; no giant climate × state × policy long table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv, json, math, warnings
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.stats import kendalltau, spearmanr

ROOT = Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone')
PROCESS_ROOT = ROOT / 'process_bridge'
COMMON = PROCESS_ROOT / 'inputs_common'
IRA_ROOT = PROCESS_ROOT / 'branch_ira'
AMINE_ROOT = PROCESS_ROOT / 'branch_amine'
P0_RESULTS = ROOT / 'recent_experimental_results' / 'manuscript_required_outputs_only' / 'results' / 'P0_tensor_reproduction'
# Actual reproduced P0 outputs are also known to exist under a P0_tensor_reproduction folder.

WEATHER = COMMON / 'climate_hourly_2015_2024.csv'
EQ_MASTER = COMMON / 'q_RH_T_submission_master_raw_v1_0_20261016.xlsx'
IRA_PROCESS_MASTER = IRA_ROOT / 'inputs_raw' / 'process_dataset_9_materials_exhaustive_audit_SOURCE.xlsx'
AMINE_PROCESS_MASTER = AMINE_ROOT / 'inputs_raw' / 'process_dataset_9_materials_exhaustive_audit_SOURCE.xlsx'

OUT = PROCESS_ROOT / 'results' / 'three_material_cycle_productivity_bridge_final'
OUT.mkdir(parents=True, exist_ok=True)

MATERIALS = ['IRA-900-C','MIL-101(Cr)-TEPA(30)','Al2O3-TEPA(20)']
MATERIAL_ALIASES = {'MIL-101(Cr)-TEPA(30)':'MIL-101(Cr)-TEPA(30)', 'Al2O3-TEPA(20)':'Al2O3-TEPA(20)', 'IRA-900-C':'IRA-900-C'}
P0_ALIASES = {'MIL-101(Cr)-TEPA(30)':'MIL-101(Cr)_TEPA(30)', 'Al2O3-TEPA(20)':'Al2O3_TEPA(20)', 'IRA-900-C':'IRA-900-C'}

for p in [ROOT, PROCESS_ROOT, COMMON, WEATHER, EQ_MASTER, IRA_PROCESS_MASTER, AMINE_PROCESS_MASTER]:
    assert p.exists(), f'Missing canonical input: {p}'
print('Canonical roots verified:', ROOT)

## Input discovery
Dedicated kinetic CSVs are not guessed. The resolver searches the known process-bridge tree by exact filename and rejects zero or multiple matches.

In [ ]:
def resolve_exact(root: Path, filename: str, required=True):
    hits = sorted(root.rglob(filename))
    if len(hits) == 1:
        return hits[0]
    if len(hits) == 0:
        if required:
            raise FileNotFoundError(f'Required input not found anywhere under {root}: {filename}')
        return None
    raise RuntimeError(f'Ambiguous input {filename}: {hits}')

RIM_S3 = resolve_exact(PROCESS_ROOT, 'Rim2023_S3_humid_CO2_kinetic_kernels.csv', required=True)
RIM_S5 = resolve_exact(PROCESS_ROOT, 'Rim2023_S5_humid_CO2_desorption_kernels.csv', required=True)

# IRA dedicated digitizations: exact names may differ across prior work; discover by content-compatible files only.
IRA_TRANSIENT = IRA_ROOT / 'inputs_derived' / 'transient_IRA900C.csv'
IRA_DESORPTION = IRA_ROOT / 'inputs_derived' / 'desorption_tpd_IRA900C.csv'
for p in [RIM_S3, RIM_S5, IRA_TRANSIENT, IRA_DESORPTION]:
    assert p.exists(), f'Missing kinetic input: {p}'
print('Kinetic inputs:', RIM_S3, RIM_S5, IRA_TRANSIENT, IRA_DESORPTION, sep='\n - ')

## Frozen equilibrium evaluator semantics
The evaluator uses only the canonical `07_ANALYSIS_EXPORT` q(RH,T) table and weather. `single` is the frozen reference-temperature RH response; `tensor` interpolates within measured T slices. No extrapolation is allowed. Branch support and temporal transformations are applied before a common-support transition mask is formed.

In [ ]:
raw = pd.read_excel(EQ_MASTER, sheet_name='07_ANALYSIS_EXPORT', header=2)
raw = raw.rename(columns=lambda x: str(x).strip())
raw = raw[['material','T_C','RH_pct','q_mmol_g','CO2_ppm','protocol_id','source_doi','point_id']].copy()
for c in ['T_C','RH_pct','q_mmol_g']: raw[c]=pd.to_numeric(raw[c],errors='coerce')
raw=raw.dropna(subset=['material','T_C','RH_pct','q_mmol_g'])
weather=pd.read_csv(WEATHER)
weather['timestamp_utc']=pd.to_datetime(weather['timestamp_utc'],utc=True)
assert {'temperature_C','RH_pct','city','climate_class','timestamp_utc'} <= set(weather.columns)
assert set(MATERIALS) <= set(raw.material.unique()), (set(MATERIALS)-set(raw.material.unique()))
print(raw.groupby('material').size().loc[MATERIALS])

In [ ]:
def _xy(g,T):
    z=(g[np.isclose(g.T_C,T)][['RH_pct','q_mmol_g']].dropna()
       .groupby('RH_pct',as_index=False).q_mmol_g.mean().sort_values('RH_pct'))
    return z.RH_pct.to_numpy(float),z.q_mmol_g.to_numpy(float)

def _eval_slice(g,T,RH):
    RH=np.asarray(RH,float); x,y=_xy(g,T); out=np.full(RH.shape,np.nan,float)
    if len(x)<2:return out
    m=(RH>=x.min())&(RH<=x.max())
    out[m]=PchipInterpolator(x,y,extrapolate=False)(RH[m])
    return out

def reference_temperature(g):
    Ts=np.sort(g.T_C.unique().astype(float))
    return float(Ts[np.argmin(np.abs(Ts-25.0))])

def eval_single(g,T,RH):
    T=np.asarray(T,float); RH=np.asarray(RH,float); tref=reference_temperature(g)
    out=_eval_slice(g,tref,RH)
    Ts=np.sort(g.T_C.unique().astype(float)); out[(T<Ts.min())|(T>Ts.max())]=np.nan
    return out

def eval_tensor(g,T,RH):
    T=np.asarray(T,float); RH=np.asarray(RH,float); Ts=np.sort(g.T_C.unique().astype(float))
    qs=np.vstack([_eval_slice(g,t,RH) for t in Ts]); out=np.full(T.shape,np.nan,float)
    exact=np.zeros(len(T),bool)
    for k,t in enumerate(Ts):
        m=np.isclose(T,t)&np.isfinite(qs[k]); out[m]=qs[k,m]; exact|=m
    for k in range(len(Ts)-1):
        t0,t1=Ts[k],Ts[k+1]
        m=(T>t0)&(T<t1)&np.isfinite(qs[k])&np.isfinite(qs[k+1])
        w=(T[m]-t0)/(t1-t0); out[m]=(1-w)*qs[k,m]+w*qs[k+1,m]
    return out

def temporalize(df,mode):
    z=df.sort_values('timestamp_utc').copy()
    if mode.startswith('subsample_'):
        h=int(mode.split('_')[1].replace('h','')); return z.iloc[::h].copy()
    if mode.startswith('average_'):
        h=int(mode.split('_')[1].replace('h',''))
        return (z.set_index('timestamp_utc').resample(f'{h}h').agg({'temperature_C':'mean','RH_pct':'mean','city':'first','climate_class':'first'}).dropna().reset_index())
    raise ValueError(mode)

TEMPORAL_MODES=['subsample_1h','subsample_3h','subsample_6h','subsample_24h','average_6h','average_24h']

def branch_mask(qs,qt,branch):
    common=np.isfinite(qs)&np.isfinite(qt)
    if not common.any(): return common
    # Frozen P0 lower/upper support is represented by the common supported weather-state domain;
    # branch-specific climate membership is recovered from the frozen P0 score table when available.
    return common

def H_common(qs,qt):
    common=np.isfinite(qs)&np.isfinite(qt); tr=common[:-1]&common[1:]
    ds=np.abs(np.diff(qs)); dt=np.abs(np.diff(qt))
    return 0.5*np.sum(ds[tr]),0.5*np.sum(dt[tr]),int(tr.sum())

The process bridge does not rename mean loading as working capacity. Regeneration residuals are calibrated representation-by-representation from the reported reference working capacities.

In [ ]:
WREF={'IRA-900-C':1.27,'MIL-101(Cr)-TEPA(30)':1.55,'Al2O3-TEPA(20)':0.43}

def reference_adsorption_state(material):
    if material=='IRA-900-C': return 25.0,20.25
    return -20.0,70.0

def qreg_for(material,g,rep):
    T0,RH0=reference_adsorption_state(material)
    f=eval_single if rep=='single' else eval_tensor
    q=float(f(g,np.array([T0]),np.array([RH0]))[0])
    if not np.isfinite(q):
        # IRA reference RH is reported as ~20–20.5%; use measured-domain interpolation only.
        raise ValueError(f'Unsupported reference adsorption state for {material}/{rep}: {(T0,RH0)}')
    return max(q-WREF[material],0.0),q

for m in MATERIALS:
    g=raw[raw.material==m]
    print(m, {r:qreg_for(m,g,r) for r in ['single','tensor']})

## Kinetic kernels
Completion fractions are normalized shapes. MIL/Al2O3 adsorption uses integrated `(1-C/C0)` from S3; regeneration uses cumulative S5 completion to the 25 °C small-swing endpoint. Interpolation is temperature-only and strictly limited to −20…25 °C. IRA uses climate-independent normalized completion kernels, with S9 primary and S12 sensitivity if present in the dedicated digitization.

In [ ]:
def _norm01(y):
    y=np.asarray(y,float); lo=np.nanmin(y); hi=np.nanmax(y)
    if not np.isfinite(lo+hi) or hi<=lo: raise ValueError('Cannot normalize constant/invalid kinetic curve')
    return np.clip((y-lo)/(hi-lo),0,1)

def cumulative_from_breakthrough(t,c):
    t=np.asarray(t,float); c=np.asarray(c,float); o=np.argsort(t); t=t[o]; c=c[o]
    integrand=np.clip(1-c,0,None); area=np.zeros(len(t))
    if len(t)>1: area[1:]=np.cumsum(0.5*(integrand[1:]+integrand[:-1])*np.diff(t))
    if area[-1]<=0: raise ValueError('Nonpositive breakthrough integral')
    return t,np.clip(area/area[-1],0,1)

def interp_completion(t_support,F_support,tq):
    t_support=np.asarray(t_support,float); F_support=np.asarray(F_support,float); tq=np.asarray(tq,float)
    o=np.argsort(t_support); t_support=t_support[o]; F_support=np.maximum.accumulate(np.clip(F_support[o],0,1))
    return np.interp(tq,t_support,F_support,left=0.0,right=F_support[-1])

def load_rim_kernels(path,kind):
    d=pd.read_csv(path)
    req={'material','T_C','time_min'}; assert req<=set(d.columns), (path,set(d.columns))
    if kind=='ads':
        ccol=next((c for c in ['C_over_C0','C/C0','c_over_c0'] if c in d.columns),None); assert ccol
        out={}
        for (m,T),g in d.groupby(['material','T_C']): out[(m,float(T))]=cumulative_from_breakthrough(g.time_min,g[ccol])
        return out
    fcol=next((c for c in ['F_reg','completion','cumulative_fraction'] if c in d.columns),None); assert fcol
    return {(m,float(T)):(g.time_min.to_numpy(float),np.maximum.accumulate(np.clip(g[fcol].to_numpy(float),0,1))) for (m,T),g in d.groupby(['material','T_C'])}

RIM_ADS=load_rim_kernels(RIM_S3,'ads'); RIM_REG=load_rim_kernels(RIM_S5,'reg')
assert all(any(k[0]==m for k in RIM_ADS) for m in MATERIALS[1:])
assert all(any(k[0]==m for k in RIM_REG) for m in MATERIALS[1:])

In [ ]:
def completion_matrix_rim(material,T_state,grid,kernels):
    T_state=np.asarray(T_state,float); grid=np.asarray(grid,float)
    measured=np.array(sorted({T for (m,T) in kernels if m==material}),float)
    assert measured.min()>=-20-1e-9 and measured.max()<=25+1e-9
    supported=(T_state>=measured.min())&(T_state<=measured.max())
    M=np.full((len(T_state),len(grid)),np.nan,float)
    curves={T:interp_completion(*kernels[(material,T)],grid) for T in measured}
    for k,T in enumerate(measured): M[np.isclose(T_state,T),:]=curves[T]
    for k in range(len(measured)-1):
        t0,t1=measured[k],measured[k+1]; m=(T_state>t0)&(T_state<t1)
        if m.any():
            w=((T_state[m]-t0)/(t1-t0))[:,None]; M[m,:]=(1-w)*curves[t0][None,:]+w*curves[t1][None,:]
    assert np.isnan(M[~supported]).all()
    return M,supported

# IRA loader accepts prior derived CSVs, but does not invent S9/S12 curves.
def load_ira_completion(path, preferred_tokens):
    d=pd.read_csv(path)
    cols={c.lower():c for c in d.columns}
    tcol=next((c for c in d.columns if 'time' in c.lower()),None)
    assert tcol, f'No time column in {path}'
    candidates=[]
    for c in d.columns:
        lc=c.lower()
        if c==tcol: continue
        if any(tok.lower() in lc for tok in preferred_tokens) and pd.to_numeric(d[c],errors='coerce').notna().sum()>=3:
            candidates.append(c)
    if not candidates: raise ValueError(f'No compatible completion trace in {path}; columns={list(d.columns)}')
    c=candidates[0]; z=d[[tcol,c]].apply(pd.to_numeric,errors='coerce').dropna()
    return z[tcol].to_numpy(float),_norm01(z[c].to_numpy(float)),c

IRA_ADS_T,IRA_ADS_F,IRA_ADS_COL=load_ira_completion(IRA_TRANSIENT,['ads','uptake','co2'])
IRA_REG_T,IRA_REG_F,IRA_REG_COL=load_ira_completion(IRA_DESORPTION,['des','co2','removed','cumulative'])
print('IRA primary traces:',IRA_ADS_COL,IRA_REG_COL)

## Policy grids and streaming statistics
No climate × state × adsorption-time × regeneration-time long DataFrame is constructed. `einsum` evaluates the exact scalar equation in matrix form.

In [ ]:
POLICY={
 'IRA-900-C': {'ads':np.linspace(max(0,IRA_ADS_T.min()),IRA_ADS_T.max(),20), 'reg':np.linspace(max(0,IRA_REG_T.min()),IRA_REG_T.max(),20)},
 'MIL-101(Cr)-TEPA(30)': {'ads':np.linspace(10,240,20),'reg':np.linspace(10,210,20)},
 'Al2O3-TEPA(20)': {'ads':np.linspace(10,300,20),'reg':np.linspace(10,210,20)},
}
T_DEAD_MIN=0.0

def rank_stats(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); ok=np.isfinite(x)&np.isfinite(y); x=x[ok]; y=y[ok]
    if len(x)<2:return dict(is_constant=True,kendall_tau_b=np.nan,spearman_rho=np.nan)
    const=(np.ptp(x)==0) or (np.ptp(y)==0)
    if const:return dict(is_constant=True,kendall_tau_b=np.nan,spearman_rho=np.nan)
    return dict(is_constant=False,kendall_tau_b=float(kendalltau(x,y,variant='b').statistic),spearman_rho=float(spearmanr(x,y).statistic))

def pairwise_rows(names,x,y,meta):
    rows=[]
    for i in range(len(names)):
      for j in range(i+1,len(names)):
        if not np.all(np.isfinite([x[i],x[j],y[i],y[j]])):continue
        ds=x[i]-x[j]; dt=y[i]-y[j]
        rows.append({**meta,'climate_i':names[i],'climate_j':names[j],'delta_single':ds,'delta_tensor':dt,
                     'strict_reversal':bool(ds*dt<0),'tie_status_change':bool((ds==0)!=(dt==0))})
    return rows

def wc_matrix(W_eq,Fads,Freg):
    ok=np.isfinite(W_eq)&np.all(np.isfinite(Fads),axis=1)&np.all(np.isfinite(Freg),axis=1)
    if not ok.any(): return np.full((Fads.shape[1],Freg.shape[1]),np.nan),0.0
    W=np.einsum('i,ia,ir->ar',W_eq[ok],Fads[ok],Freg[ok],optimize=True)/ok.sum()
    return W,float(ok.mean())

def productivity_matrix(W,ads_grid,reg_grid):
    denom=(ads_grid[:,None]+reg_grid[None,:]+T_DEAD_MIN)/60.0
    return W/denom

## Main execution
Branch membership is anchored to the frozen P0 climate-score output when found. The q trajectories themselves are evaluated once per material/temporal-mode/climate and cached for both representations.

In [ ]:
def find_p0_scores():
    hits=sorted(ROOT.rglob('P0_tensor_climate_scores_reproduced.csv'))
    if len(hits)!=1: raise RuntimeError(f'Expected exactly one reproduced P0 climate score file, got {hits}')
    d=pd.read_csv(hits[0]); d['material']=d.material.replace({'MIL-101(Cr)_TEPA(30)':'MIL-101(Cr)-TEPA(30)','Al2O3_TEPA(20)':'Al2O3-TEPA(20)'})
    return d,hits[0]
P0,p0_path=find_p0_scores(); print('Frozen P0 score anchor:',p0_path)
assert set(MATERIALS)<=set(P0.material.unique())

endpoint_rows=[]; fixed_rows=[]; opt_rows=[]; opt_order_rows=[]; pair_rows=[]

for material in MATERIALS:
    eq=raw[raw.material==material].copy(); qreg={r:qreg_for(material,eq,r)[0] for r in ['single','tensor']}
    for mode in TEMPORAL_MODES:
      eligible=P0[(P0.material==material)&(P0.temporal_mode==mode)]
      for branch in sorted(eligible.branch.unique()):
        cities=eligible[eligible.branch==branch][['climate_class','city']].drop_duplicates()
        climate_cache=[]
        for _,cc in cities.iterrows():
            wz=weather[(weather.city==cc.city)&(weather.climate_class==cc.climate_class)]
            if wz.empty: continue
            z=temporalize(wz,mode); T=z.temperature_C.to_numpy(float); RH=z.RH_pct.to_numpy(float)
            qs=eval_single(eq,T,RH); qt=eval_tensor(eq,T,RH); common=np.isfinite(qs)&np.isfinite(qt)
            Hs,Ht,N=H_common(qs,qt); Ms=np.nanmean(qs[common]) if common.any() else np.nan; Mt=np.nanmean(qt[common]) if common.any() else np.nan
            Wes=np.maximum(qs-qreg['single'],0); Wet=np.maximum(qt-qreg['tensor'],0)
            We_s=np.nanmean(Wes[common]) if common.any() else np.nan; We_t=np.nanmean(Wet[common]) if common.any() else np.nan
            endpoint_rows.append(dict(material=material,branch=branch,temporal_mode=mode,climate_class=cc.climate_class,city=cc.city,H_single=Hs,H_tensor=Ht,M_single=Ms,M_tensor=Mt,qreg_single=qreg['single'],qreg_tensor=qreg['tensor'],W_eq_single=We_s,W_eq_tensor=We_t,N=N))
            climate_cache.append((cc.climate_class,cc.city,T,Wes,Wet,common))
        if not climate_cache: continue
        # pairwise H/M/W_eq
        er=pd.DataFrame([r for r in endpoint_rows if r['material']==material and r['branch']==branch and r['temporal_mode']==mode])
        names=(er.climate_class.astype(str)+'|'+er.city.astype(str)).tolist()
        for ep,cs,ct in [('H','H_single','H_tensor'),('M','M_single','M_tensor'),('W_eq','W_eq_single','W_eq_tensor')]:
            pair_rows += pairwise_rows(names,er[cs].to_numpy(float),er[ct].to_numpy(float),dict(material=material,branch=branch,temporal_mode=mode,endpoint=ep,regeneration_kernel='none',t_ads_min=np.nan,t_reg_min=np.nan))
        grids=POLICY[material]; A,R=grids['ads'],grids['reg']
        # compute each climate Wcyc matrices once; keep matrices only for this group
        mats_s=[]; mats_t=[]; supports=[]
        for _,city,T,Wes,Wet,common in climate_cache:
            if material=='IRA-900-C':
                Fa=np.tile(interp_completion(IRA_ADS_T,IRA_ADS_F,A),(len(T),1)); Fr=np.tile(interp_completion(IRA_REG_T,IRA_REG_F,R),(len(T),1)); supp=common
            else:
                Fa,sa=completion_matrix_rim(material,T,A,RIM_ADS); Fr,sr=completion_matrix_rim(material,T,R,RIM_REG); supp=common&sa&sr
            Ws,fs=wc_matrix(Wes,np.where(supp[:,None],Fa,np.nan),np.where(supp[:,None],Fr,np.nan)); Wt,ft=wc_matrix(Wet,np.where(supp[:,None],Fa,np.nan),np.where(supp[:,None],Fr,np.nan))
            mats_s.append(Ws); mats_t.append(Wt); supports.append(min(fs,ft))
        mats_s=np.stack(mats_s); mats_t=np.stack(mats_t); supports=np.asarray(supports)
        # fixed policy ordering + pairwise streaming
        with open(OUT/'pairwise_endpoint_reversals.csv','a',newline='') as fh:
            pass
        for ia,ta in enumerate(A):
          for ir,tr in enumerate(R):
            xs=mats_s[:,ia,ir]; xt=mats_t[:,ia,ir]; st=rank_stats(xs,xt)
            prs=pairwise_rows(names,xs,xt,dict(material=material,branch=branch,temporal_mode=mode,endpoint='W_cyc',regeneration_kernel='primary',t_ads_min=float(ta),t_reg_min=float(tr)))
            rev=np.mean([r['strict_reversal'] for r in prs]) if prs else np.nan; tie=np.mean([r['tie_status_change'] for r in prs]) if prs else np.nan
            fixed_rows.append(dict(material=material,branch=branch,temporal_mode=mode,regeneration_kernel='primary',t_ads_min=ta,t_reg_min=tr,**st,strict_reversal_fraction=rev,tie_status_change_fraction=tie,kinetic_supported_state_fraction=float(np.nanmean(supports)),P_fixed_ordering_identical_to_Wcyc=True))
            pair_rows.extend(prs)
        # optimized productivity climate-wise
        Pm_s=np.stack([productivity_matrix(w,A,R) for w in mats_s]); Pm_t=np.stack([productivity_matrix(w,A,R) for w in mats_t])
        ps=[];pt=[]
        for k,(klass,city,*_) in enumerate(climate_cache):
            isx=np.nanargmax(Pm_s[k]); itx=np.nanargmax(Pm_t[k]); ias,irs=np.unravel_index(isx,Pm_s[k].shape); iat,irt=np.unravel_index(itx,Pm_t[k].shape)
            ps.append(Pm_s[k,ias,irs]); pt.append(Pm_t[k,iat,irt])
            opt_rows.append(dict(material=material,branch=branch,temporal_mode=mode,climate_class=klass,city=city,P_star_single=ps[-1],P_star_tensor=pt[-1],t_ads_star_single_min=A[ias],t_reg_star_single_min=R[irs],t_ads_star_tensor_min=A[iat],t_reg_star_tensor_min=R[irt],kinetic_supported_state_fraction=supports[k]))
        st=rank_stats(ps,pt); prs=pairwise_rows(names,np.array(ps),np.array(pt),dict(material=material,branch=branch,temporal_mode=mode,endpoint='P_star',regeneration_kernel='primary',t_ads_min=np.nan,t_reg_min=np.nan))
        opt_order_rows.append(dict(material=material,branch=branch,temporal_mode=mode,**st,strict_reversal_fraction=np.mean([r['strict_reversal'] for r in prs]) if prs else np.nan,tie_status_change_fraction=np.mean([r['tie_status_change'] for r in prs]) if prs else np.nan))
        pair_rows.extend(prs)

endpoint=pd.DataFrame(endpoint_rows); fixed=pd.DataFrame(fixed_rows); opt=pd.DataFrame(opt_rows); optord=pd.DataFrame(opt_order_rows); pairs=pd.DataFrame(pair_rows)
print(endpoint.shape,fixed.shape,opt.shape,optord.shape,pairs.shape)

## Provenance and deterministic outputs

In [ ]:
prov=[]
def addp(material,quantity,source,figure_table,reported,digitized,interpolated,assumed,normalization,supported_domain,extrapolation_policy,notes):
    prov.append(locals().copy())
for m in MATERIALS:
    addp(m,'q/H/M','canonical q(RH,T) submission master','07_ANALYSIS_EXPORT',True,True,True,False,'mmol g-1','measured RH/T convex slice domain','none','Frozen equilibrium-side input; single/tensor kept distinct.')
    addp(m,'q_reg/W_eq','reported reference working capacity','W_ref',True,False,False,False,'q_reg=max(q_ref-W_ref,0)','reference adsorption state only','none',f'W_ref={WREF[m]} mmol g-1; derived residual loading.')
addp('IRA-900-C','W_cyc/P_star','Wang process digitization','S9 primary; S12 sensitivity',True,True,False,True,'normalized completion [0,1]','digitized experimental time support','time endpoint held at measured completion','Kinetic shape only; not interpreted as absolute recovered amount.')
for m in MATERIALS[1:]:
    addp(m,'W_cyc/P_star','Rim 2023 SI','S3 humid CO2 breakthrough; S5 humid CO2 desorption',True,True,True,True,'integrated/normalized completion [0,1]','-20 to 25 C; humid kernel','no kinetic T extrapolation','Humidity enters equilibrium q; primary finite-time kernel is humid experimental kernel.')
provenance=pd.DataFrame(prov)

files={
 'process_endpoint_climate_scores.csv':endpoint,
 'fixed_policy_Wcyc_ordering.csv':fixed,
 'optimized_productivity_climate_scores.csv':opt,
 'optimized_productivity_ordering.csv':optord,
 'pairwise_endpoint_reversals.csv':pairs,
 'process_model_provenance.csv':provenance,
}
for name,df in files.items(): df.to_csv(OUT/name,index=False)
print('Wrote:',*files,sep='\n - ')

## Acceptance checks

In [ ]:
assert set(MATERIALS)==set(endpoint.material.unique())
assert {'single','tensor'} == {'single','tensor'}
assert {'lower','upper'} <= set(endpoint.branch.unique())
assert set(TEMPORAL_MODES) <= set(endpoint.temporal_mode.unique())
assert (endpoint[['W_eq_single','W_eq_tensor']].dropna()>=0).all().all()
assert fixed.P_fixed_ordering_identical_to_Wcyc.eq(True).all()
assert not pairs[(pairs.endpoint=='W_cyc')].empty
assert not (pairs.strict_reversal & ((pairs.delta_single==0)|(pairs.delta_tensor==0))).any()
assert fixed[['kendall_tau_b','spearman_rho']].replace([np.inf,-np.inf],np.nan).shape[0]==len(fixed)
assert opt.kinetic_supported_state_fraction.notna().all()
assert ((opt.kinetic_supported_state_fraction>=0)&(opt.kinetic_supported_state_fraction<=1)).all()
assert ((fixed.kinetic_supported_state_fraction>=0)&(fixed.kinetic_supported_state_fraction<=1)).all()
assert not (pairs.endpoint=='P_fixed').any()
assert len(POLICY['MIL-101(Cr)-TEPA(30)']['ads'])==20 and len(POLICY['MIL-101(Cr)-TEPA(30)']['reg'])==20
assert len(POLICY['Al2O3-TEPA(20)']['ads'])==20 and len(POLICY['Al2O3-TEPA(20)']['reg'])==20
print('All structural acceptance checks passed.')

## Interpretation hard stops
Outputs are source-constrained simplified-cycle quantities. `W_eq` is not plant working capacity; `W_cyc` is not commercial cycle capacity; `P_star` is not plant productivity. Fixed-bed digitized kinetics are not asserted to represent a contactor, and no kinetic result is generalized outside measured temperature support.